In [1]:
import hashlib

In [2]:
BASE58_ALPHABET = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'

In [3]:
def _double_sha256(data : bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

In [4]:
def b58check_encode(version : bytes,playload: bytes)-> str:
    data = version + playload

    checksum = _double_sha256(data)[:4]
    return b58encode(data + checksum)

In [5]:
def b58check_decode(base58_str: str) -> tuple[bytes, bytes]:
    raw_data = b58decode(base58_str)

    if len(raw_data)<5:
        raise ValueError('Decoded Data is too short to contain a valid checksum')
    data = raw_data[:-4]
    provided_checksum = raw_data[-4:]

    expected_checksum = _double_sha256(data)[:4]
    if provided_checksum != expected_checksum:
        raise ValueError('invalid Base58check checksum ! data is corrupt or mistyped.')
    version = data[:1]
    playload = data[1:]
    return version,playload 


In [6]:
def b58encode(data: bytes) -> str:
    leading_zeros = len(data) - len(data.lstrip(b"\x00"))
    num = int.from_bytes(data, "big")
    encoded = ""

    while num:
        num, remainder = divmod(num, 58)
        encoded = BASE58_ALPHABET[remainder] + encoded

    return "1" * leading_zeros + encoded


def b58decode(address: str) -> bytes:
    num = 0

    for char in address:
        if char not in BASE58_ALPHABET:
            raise ValueError(f"Invalid character '{char}' in Base58 string")
        num = num * 58 + BASE58_ALPHABET.index(char)

    num_bytes = num.to_bytes((num.bit_length() + 7) // 8, "big") if num else b""
    leading_ones = len(address) - len(address.lstrip("1"))
    return b"\x00" * leading_ones + num_bytes

In [7]:
if __name__ == "__main__":

    version_byte = b"\x00"
    
    mock_hash160 = bytes.fromhex("f54a5851e9372b87810a8e60cdd2e7cfd80b6e31")
    
    address = b58check_encode(version_byte, mock_hash160)
    print(f"Encoded Address: {address}")

    try:
        ver, payload = b58check_decode(address)
        print(f"Validation Result: Success!")
        print(f"Version Byte: 0x{ver.hex()}")
        print(f"Payload: {payload.hex()}")
    except ValueError as e:
        print(f"Validation Failed: {e}")
    
    corrupted_address = address[:-1] + ("x" if address[-1] != "x" else "y")
    print(f"\nTesting Corrupted Address: {corrupted_address}")
    try:
        b58check_decode(corrupted_address)
    except ValueError as e:
        print(f"Caught expected error: {e}")

Encoded Address: 1PMycacnJaSqwwJqjawXBErnLsZ7RkXUAs
Validation Result: Success!
Version Byte: 0x00
Payload: f54a5851e9372b87810a8e60cdd2e7cfd80b6e31

Testing Corrupted Address: 1PMycacnJaSqwwJqjawXBErnLsZ7RkXUAx
Caught expected error: invalid Base58check checksum ! data is corrupt or mistyped.
